In [ ]:
import os
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


os.chdir(os.path.dirname(os.getcwd()))


In [ ]:
df = pd.read_csv("data/processed/portfolio.csv")

In [ ]:
# Convert 'Date' to datetime
df["Date"] = pd.to_datetime(df["Date"])

# Create a Month-Year column
df["MonthYear"] = df["Date"].dt.to_period("M")

# Find last working day of each month
last_working_days = df.groupby("MonthYear")["Date"].apply(
    lambda x: x[x.dt.is_month_end & ~x.dt.dayofweek.isin([5, 6])].max()
)

# Filter original dataframe to only include rows from last working day
df_filtered = df[df["Date"].isin(last_working_days.values)]

# Optional: drop MonthYear if you want to reassign it based on filtered dates
df_filtered["MonthYear"] = df_filtered["Date"].dt.to_period("M").astype(str)

In [ ]:
# Convert 'Date' to datetime and extract Month-Year
df["Date"] = pd.to_datetime(df["Date"])
df["MonthYear"] = df["Date"].dt.to_period("M")

# Find last calendar day of each month
last_days = df.groupby("MonthYear")["Date"].max()

# Filter to rows from last day of each month
df_filtered = df[df["Date"].isin(last_days.values)]

# Pivot for stacked bar chart
pivot_df = df_filtered.pivot_table(
    index="Date", columns="Product", values="eur", aggfunc="sum", fill_value=0
)

# Create subplot figure
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.15,
    subplot_titles=[
        "EUR per Product on Last Day of Month",
        "Total EUR on Last Day of Month",
    ],
)

# Stacked bar chart per product
for product in pivot_df.columns:
    fig.add_trace(
        go.Bar(x=pivot_df.index, y=pivot_df[product], name=product), row=1, col=1
    )

# Total EUR per month
monthly_total = pivot_df.sum(axis=1)
fig.add_trace(
    go.Line(
        x=monthly_total.index,
        y=monthly_total.values,
        name="Total EUR",
        marker_color="gray",
    ),
    row=2,
    col=1,
)

# Layout settings
fig.update_layout(
    barmode="stack",
    height=700,
    xaxis_title="Month-Year",
    legend_title="Product",
    template="plotly_white",
)

fig.update_yaxes(title_text="EUR per Product", row=1, col=1)
fig.update_yaxes(title_text="Total EUR", row=2, col=1)

fig.show()